In [1]:
# Imports and path setup
import pandas as pd
from pathlib import Path
import sys
sys.path.insert(0, "src")

import importlib
import ingest
import qc_checks
import kpi
import api_ingest

for module in [ingest, qc_checks, kpi, api_ingest]:
    print(importlib.reload(module))


<module 'ingest' from '/Users/mansi/Documents/Projects/esg-emissions-pipeline/src/ingest.py'>
<module 'qc_checks' from '/Users/mansi/Documents/Projects/esg-emissions-pipeline/src/qc_checks.py'>
<module 'kpi' from '/Users/mansi/Documents/Projects/esg-emissions-pipeline/src/kpi.py'>
<module 'api_ingest' from '/Users/mansi/Documents/Projects/esg-emissions-pipeline/src/api_ingest.py'>


In [2]:
# Step 1: Ingest raw data from choosen data source
# Set DATA_SOURCE to "owid" (static CSV) or "api" (live World Bank REST API).

DATA_SOURCE = "api"  # ENTER "api" to use the World Bank API
                     # ENTER "owid" to use OWID dataset. 

if DATA_SOURCE == "owid":
    landing_path = "data/raw/portfolio_landing.csv"
    df = ingest.run(raw_path="data/raw/owid-co2-data.csv", landing_path=landing_path)
    print(f"[OWID] Ingested {len(df)} rows across {df['country'].nunique()} entities.")
elif DATA_SOURCE == "api":
    landing_path = "data/raw/portfolio_landing_api.csv"
    df = api_ingest.run(landing_path=landing_path)
    print(f"[World Bank API] Fetched {len(df)} rows across {df['country'].nunique()} entities.")
else:
    raise ValueError("DATA_SOURCE must be 'owid' or 'api'")

df.head()

[World Bank API] Fetched 240 rows across 10 entities.


,country,year,co2,total_ghg,gdp
0,Brazil,2000,349.4076,870.3341,6.554482e+11
1,Brazil,2001,351.3367,882.1005,5.599836e+11
2,Brazil,2002,351.4370,904.5260,5.097953e+11
3,Brazil,2003,346.5055,934.4198,5.582337e+11
4,Brazil,2004,365.3106,983.4187,6.692894e+11


In [3]:
# Step2: QA/QC

summary = qc_checks.run(
    landing_path="data/raw/portfolio_landing.csv",
    processed_path="data/processed/portfolio_clean.csv",
    errors_path="data/errors/portfolio_rejects.csv",
    validity_path="data/processed/kpis/kpi_validity_by_entity.csv"
)
summary

{'total_records': 240,
 'passed': 222,
 'rejected': 18,
 'completeness_score': np.float64(0.997),
 'overall_validity_score': 0.925}

In [4]:
# Step3: Calculate KPIs

kpi.run(
    processed_path="data/processed/portfolio_clean.csv",
    output_dir="data/processed/kpis",
)

KPIs written to data/processed/kpis (latest year: 2023)


In [6]:
# Inspecting KPI outputs

kpi_dir = Path("data/processed/kpis")

for file in sorted(kpi_dir.glob("*.csv")):
    print(f"--- {file.name} ---")
    display(pd.read_csv(file))

--- kpi_completeness_by_entity.csv ---


,country,completeness_score
0,Brazil,0.9965
1,China,0.9960
2,Denmark,0.9962
3,Germany,1.0000
4,India,0.9967
5,Netherlands,0.9967
6,Norway,0.9968
7,Sweden,0.9965
8,United Kingdom,0.9967
9,United States,0.9967


--- kpi_intensity_gdp.csv ---


,country,year,co2_per_gdp_million
0,Brazil,2000,0.000196
1,Brazil,2001,0.000194
2,Brazil,2002,0.000187
3,Brazil,2003,0.000181
4,Brazil,2004,0.000177
...,...,...,...
217,United States,2018,0.000296
218,United States,2019,0.000282
219,United States,2021,0.000263
220,United States,2022,0.000259


--- kpi_portfolio_ghg_total.csv ---


...

,country,co2
0,China,12172.009
1,United States,4918.407
2,India,3062.756
3,Brazil,483.992
4,United Kingdom,307.826
5,Netherlands,117.016
6,Norway,38.869
7,Sweden,36.709
8,Denmark,28.831


--- kpi_validity_by_entity.csv ---


,country,validity_score
0,Brazil,0.9167
1,China,0.7917
2,Denmark,0.8333
3,Germany,0.9583
4,India,0.9583
5,Netherlands,0.9583
6,Norway,1.0000
7,Sweden,0.9167
8,United Kingdom,0.9583
9,United States,0.9583


--- kpi_yoy_change.csv ---


,country,year,yoy_change_pct
0,Brazil,2000,NaN
1,Brazil,2001,1.758759
2,Brazil,2002,0.461917
3,Brazil,2003,-0.897158
4,Brazil,2004,4.871389
...,...,...,...
217,United States,2018,3.191640
218,United States,2019,-2.337595
219,United States,2021,-4.121555
220,United States,2022,0.703012
